In [ ]:
import os
import sys
import torch
import pandas as pd
from pathlib import Path

# 确保当前目录在搜索路径中，这样才能 import models
sys.path.append(os.getcwd())

from preprocess import DemuxManager
from classifier import HTClassifier
from evaluator import evaluate_demux 
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.getcwd())

In [ ]:
# 1. 初始化管理器
manager = DemuxManager()

# 2. 设置路径
data_dir = Path(r"/home/chongxiao/HT_Demux/HTO016")
mtx = data_dir / "matrix.mtx.gz"
feat = data_dir / "features.tsv.gz"
bar = data_dir / "barcodes.tsv.gz"

# 3. 加载数据
print("Loading data...")
counts, barcodes = manager.load_10x_mtx(mtx, feat, bar)
manager.build_configs(max_klet=2)

# 4. 运行模型
print("Running optimizer...")
results = manager.run_demux(counts, model_type="GMM", method="GD", device="cuda" if torch.cuda.is_available() else "cpu")

# 5. 分类并保存结果
clf = HTClassifier(results["cfg_labels"], results["tag_names"], manager.configs)
df_results = clf.classify(results["posteriors"], barcodes, map_thresh=0.1)

# 保存到本地以便 test.py 读取
output_path = "demux_results.csv"
df_results.to_csv(output_path, index=False)
print(f"Demux complete. Results saved to {output_path}")

# 预览前几行
df_results.head()

In [ ]:
# 指定 Ground Truth 路径
truth_path = r"/home/chongxiao/HT_Demux/HTO016/ground_truth_droplet.csv"

# 调用评估函数
evaluate_demux(pred_path="demux_results.csv", truth_path=truth_path)
